In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- diagnostics ---
print("CWD:", Path.cwd())

# --- ROOT / DATA_DIR / OUT_DIR ---
ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "out_data"
OUT_DIR.mkdir(exist_ok=True)

print("ROOT   :", ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR :", OUT_DIR)

# required outputs from your earlier data-construction pipeline
FEATURES_FILE = OUT_DIR / "binary_vosa_XPcoeff_110d_L2.csv"
LABELS_FILE   = DATA_DIR / "VOSA_labels_training.csv"

if not FEATURES_FILE.exists():
    raise FileNotFoundError(f"Missing features file: {FEATURES_FILE}\n"
                            "Put out_data/binary_vosa_XPcoeff_110d_L2.csv in place (from your XP build notebook).")
if not LABELS_FILE.exists():
    raise FileNotFoundError(f"Missing labels file: {LABELS_FILE}\n"
                            "Put data/VOSA_labels_training.csv in place (from the repo).")

print("OK: Found", FEATURES_FILE.name, "and", LABELS_FILE.name)


In [2]:
df_feat = pd.read_csv(FEATURES_FILE)
df_lab  = pd.read_csv(LABELS_FILE)

# labels file should have columns like: ['GaiaDR3','VOSA']
if not {"GaiaDR3", "VOSA"}.issubset(df_lab.columns):
    raise ValueError(f"Unexpected label columns: {list(df_lab.columns)} (expected GaiaDR3, VOSA)")

df_lab = df_lab.rename(columns={"GaiaDR3": "source_id", "VOSA": "y"}).copy()
df_lab["source_id"] = df_lab["source_id"].astype("int64")
df_lab["y"] = df_lab["y"].astype(int)

# features file should have: source_id, y, c000..c109
if "source_id" not in df_feat.columns:
    raise ValueError(f"Features missing source_id. Columns: {list(df_feat.columns)[:10]} ...")
if "y" not in df_feat.columns:
    df_feat["source_id"] = df_feat["source_id"].astype("int64")
    df_feat = df_feat.merge(df_lab[["source_id", "y"]], on="source_id", how="inner")

# pick coefficient columns
coef_cols = [c for c in df_feat.columns if c.startswith("c") and len(c) == 4]  # c000..c109
if len(coef_cols) != 110:
    coef_cols = [c for c in df_feat.columns if c.startswith("c")]
if len(coef_cols) != 110:
    raise ValueError(f"Expected 110 coefficient columns, found {len(coef_cols)}. Example: {coef_cols[:10]}")

coef_cols = sorted(coef_cols)
df_feat["source_id"] = df_feat["source_id"].astype("int64")
df_feat["y"] = df_feat["y"].astype(int)

# Ensure labels consistent
if "y" in df_feat.columns and "y" in df_lab.columns:
    overlap = df_feat.merge(df_lab[["source_id","y"]], on="source_id", suffixes=("_feat","_lab"))
    if (overlap["y_feat"] != overlap["y_lab"]).any():
        bad = overlap.loc[overlap["y_feat"] != overlap["y_lab"], ["source_id","y_feat","y_lab"]].head(5)
        raise ValueError(f"Label mismatch between features and labels file. Examples:\n{bad}")

X = df_feat[coef_cols].to_numpy(dtype=float)
y = df_feat["y"].to_numpy(dtype=int)

pos = int(y.sum())
neg = int((y == 0).sum())
print("Rows:", len(y), "| positives:", pos, "| neg:", neg)
print("X shape:", X.shape, "| y shape:", y.shape)
print("coef cols:", coef_cols[:5], "...", coef_cols[-5:])


Rows: 2815 | positives: 558 | neg: 2257
X shape: (2815, 110) | y shape: (2815,)
coef cols: ['c000', 'c001', 'c002', 'c003', 'c004'] ... ['c105', 'c106', 'c107', 'c108', 'c109']


In [3]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# 60/20/20 split
X_tmp, X_te, y_tmp, y_te = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tmp, y_tmp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_tmp
)  # 0.25 of 0.8 = 0.2

print("Train:", X_tr.shape, "pos:", int(y_tr.sum()))
print("Val  :", X_va.shape, "pos:", int(y_va.sum()))
print("Test :", X_te.shape, "pos:", int(y_te.sum()))


Train: (1689, 110) pos: 335
Val  : (563, 110) pos: 111
Test : (563, 110) pos: 112


In [4]:
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

def _metrics_from_probs(y_true, p_pos, thr):
    y_true = np.asarray(y_true).astype(int)
    p_pos  = np.asarray(p_pos).astype(float)

    y_pred = (p_pos >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    acc  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0
    f1   = (2 * prec * sens) / (prec + sens) if (prec + sens) else 0.0
    youden = sens + spec - 1.0
    return sens, spec, prec, acc, f1, youden

def pick_threshold(y_true, p_pos, criterion="youden", grid_size=200):
    p_pos = np.asarray(p_pos).astype(float)
    qs = np.linspace(0.0, 1.0, grid_size)
    thr_grid = np.unique(np.quantile(p_pos, qs))
    thr_grid = np.clip(thr_grid, 0.0, 1.0)


    best = None
    for thr in thr_grid:
        sens, spec, prec, acc, f1, youden = _metrics_from_probs(y_true, p_pos, thr)
        score = youden if criterion == "youden" else f1
        row = (score, float(thr), sens, spec, prec, acc)
        if best is None or row[0] > best[0]:
            best = row

    score, thr, sens, spec, prec, acc = best
    return {"thr": thr, "sens": sens, "spec": spec, "prec": prec, "acc": acc, "score": score}

def prob_metrics(y_true, p_pos):
    y_true = np.asarray(y_true).astype(int)
    p_pos  = np.asarray(p_pos).astype(float)

    # guard logloss in case of exact 0/1
    eps = 1e-15
    p_clip = np.clip(p_pos, eps, 1 - eps)

    return {
        "ROC AUC": float(roc_auc_score(y_true, p_pos)),
        "PR AUC":  float(average_precision_score(y_true, p_pos)),
        "Brier":   float(brier_score_loss(y_true, p_pos)),
        "Log loss":float(log_loss(y_true, p_clip)),
    }

def style_table_viridis(df):
    metric_cols = [c for c in [
        "Sensitivity", "Specificity", "Precision", "Accuracy",
        "ROC AUC", "PR AUC", "Brier", "Log loss"
    ] if c in df.columns]

    fmt = {}
    for c in metric_cols:
        fmt[c] = "{:.3f}"
    if "Threshold" in df.columns:
        fmt["Threshold"] = "{:.3f}"

    # common best-hyper columns (present depending on model)
    for c in df.columns:
        if c.startswith("Best "):
            fmt[c] = "{}"

    return (df.style
            .format(fmt, na_rep="")
            .background_gradient(subset=metric_cols, cmap="viridis")
            .hide(axis="index"))

print("Helpers ready.")


Helpers ready.


In [5]:
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

# --- grids ---
# Classic GBDT (slower than HGB on big grids)
GB_N_EST = [100, 300, 600]
GB_LR    = [0.03, 0.1]
GB_MAXD  = [2, 3]               # depth of individual trees
GB_SUBS  = [1.0, 0.7]           # stochastic gradient boosting when <1.0
GB_MINL  = [1, 2, 5]

# HistGB (fast)
HGB_MAX_IT = [200, 600]
HGB_LR     = [0.03, 0.1]
HGB_MAXD   = [None, 6, 10]
HGB_MINL   = [20, 50]
HGB_L2     = [0.0, 0.1]         # L2 reg on leaf values
HGB_SUBS   = [1.0, 0.7]         # subsample when <1.0 (stochastic)

# --- variants ---
VARIANTS = [
    {"name": "GBDT (classic)",             "kind": "gb",  "calibration": None},
    {"name": "GBDT (classic) + sigmoid",   "kind": "gb",  "calibration": "sigmoid"},
    {"name": "HGB (fast)",                 "kind": "hgb", "calibration": None},
    {"name": "HGB (fast) + sigmoid",       "kind": "hgb", "calibration": "sigmoid"},
    {"name": "HGB (fast) + isotonic",      "kind": "hgb", "calibration": "isotonic"},
]

print("Variants:", [v["name"] for v in VARIANTS])


Variants: ['GBDT (classic)', 'GBDT (classic) + sigmoid', 'HGB (fast)', 'HGB (fast) + sigmoid', 'HGB (fast) + isotonic']


In [6]:
def fit_model(params, variant):
    kind = variant["kind"]

    if kind == "gb":
        base = GradientBoostingClassifier(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=params["subsample"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=RANDOM_STATE,
        )
    elif kind == "hgb":
        base = HistGradientBoostingClassifier(
            max_iter=params["max_iter"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            l2_regularization=params["l2_regularization"],
            max_bins=255,
            early_stopping=False,
            random_state=RANDOM_STATE,
        )

        if "max_features" in params:
            base.set_params(max_features=params["max_features"])
    else:
        raise ValueError("Unknown variant kind")

    if variant["calibration"] is None:
        base.fit(X_tr, y_tr)
        return base

    cal = CalibratedClassifierCV(base, method=variant["calibration"], cv=3)
    cal.fit(X_tr, y_tr)
    return cal


def iter_param_grid(variant):
    if variant["kind"] == "gb":
        for n in GB_N_EST:
            for lr in GB_LR:
                for md in GB_MAXD:
                    for subs in GB_SUBS:
                        for minl in GB_MINL:
                            yield {
                                "n_estimators": int(n),
                                "learning_rate": float(lr),
                                "max_depth": int(md),
                                "subsample": float(subs),
                                "min_samples_leaf": int(minl),
                            }
    else:
        for it in HGB_MAX_IT:
            for lr in HGB_LR:
                for md in HGB_MAXD:
                    for minl in HGB_MINL:
                        for l2 in HGB_L2:
                            for mf in HGB_SUBS:
                                yield {
                                    "max_iter": int(it),
                                    "learning_rate": float(lr),
                                    "max_depth": md,  # can be None
                                    "min_samples_leaf": int(minl),
                                    "l2_regularization": float(l2),
                                    "max_features": float(mf),
                                }


def search_on_val(variant, criterion):
    best_score = None
    best_model = None
    best_params = None
    best_thr = None

    for params in iter_param_grid(variant):
        model = fit_model(params, variant)

        # probability for threshold tuning
        if hasattr(model, "predict_proba"):
            p_va = model.predict_proba(X_va)[:, 1]
        else:
            # safety, but should not happen for these
            raise RuntimeError("Model has no predict_proba; cannot tune threshold here.")

        pick = pick_threshold(y_va, p_va, criterion=criterion)
        score = pick["score"]

        if (best_score is None) or (score > best_score):
            best_score = score
            best_model = model
            best_params = params
            best_thr = pick["thr"]

    return {"model": best_model, "params": best_params, "thr": float(best_thr)}


def evaluate_on_test(model, thr):
    p_te = model.predict_proba(X_te)[:, 1]
    sens, spec, prec, acc, f1, youden = _metrics_from_probs(y_te, p_te, thr)
    return {
        "Sensitivity": float(sens),
        "Specificity": float(spec),
        "Precision": float(prec),
        "Accuracy": float(acc),
    }, prob_metrics(y_te, p_te)


In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

# ---------- Imports  ----------
missing = []
try:
    import xgboost as xgb
except Exception as e:
    missing.append(("xgboost", e))

try:
    import lightgbm as lgb
except Exception as e:
    missing.append(("lightgbm", e))

try:
    from catboost import CatBoostClassifier
except Exception as e:
    missing.append(("catboost", e))

if missing:
    msg = "\n".join([f"- {name}: {repr(err)}" for name, err in missing])
    raise ImportError(
        "Trūksta bibliotekų. Įdiek vieną kartą (tavo .venv aktyvuotas):\n"
        "  pip install xgboost lightgbm catboost\n\n"
        f"Detaliau:\n{msg}"
    )

X_tr_np = np.asarray(X_tr)
X_va_np = np.asarray(X_va)
X_te_np = np.asarray(X_te)


def style_prob_table_viridis(df: pd.DataFrame) -> "pd.io.formats.style.Styler":
    metric_cols = [c for c in ["ROC_AUC", "PR_AUC", "Brier", "LogLoss"] if c in df.columns]
    fmt = {c: "{:.4f}" for c in metric_cols}
    return (df.style
            .format(fmt, na_rep="")
            .background_gradient(subset=metric_cols, cmap="viridis")
            .hide(axis="index"))

pos = int(np.sum(y_tr == 1))
neg = int(np.sum(y_tr == 0))
scale_pos_weight = (neg / max(pos, 1))

print("Train size:", len(y_tr), "| pos:", pos, "| neg:", neg, "| scale_pos_weight:", round(scale_pos_weight, 3))

def _make_hashable(v):
    if isinstance(v, np.ndarray):
        v = v.tolist()
    if isinstance(v, dict):
        return tuple((k, _make_hashable(v[k])) for k in sorted(v))
    if isinstance(v, (list, tuple)):
        return tuple(_make_hashable(x) for x in v)
    return v

def sample_param_space(space: dict, n_samples: int, seed: int):
    """
    space: {param_name: [choices...], ...}
    returns list of dicts, unique combos, deterministic for given seed
    """
    rng = np.random.default_rng(seed)
    keys = list(space.keys())

    out = []
    seen = set()
    tries = 0
    while len(out) < n_samples and tries < n_samples * 50:
        tries += 1
        d = {k: rng.choice(space[k]) for k in keys}
        # normalize numpy containers to python types
        for k, v in d.items():
            if isinstance(v, np.ndarray):
                d[k] = v.tolist()
            elif isinstance(v, np.generic):
                d[k] = v.item()
        key = tuple((k, _make_hashable(d[k])) for k in keys)
        if key not in seen:
            seen.add(key)
            out.append(d)
    return out


EARLY_STOP = 200

def build_xgb(params: dict):
    
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        booster=params["booster"],
        eval_metric=params["eval_metric"],
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,

        n_estimators=int(params["n_estimators"]),
        max_depth=int(params["max_depth"]),
        learning_rate=float(params["learning_rate"]),
        subsample=float(params["subsample"]),
        colsample_bytree=float(params["colsample_bytree"]),

        min_child_weight=float(params["min_child_weight"]),
        gamma=float(params["gamma"]),
        reg_lambda=float(params["reg_lambda"]),
        reg_alpha=float(params["reg_alpha"]),

        scale_pos_weight=float(params["scale_pos_weight"]),
        max_delta_step=float(params["max_delta_step"]),

        rate_drop=float(params.get("rate_drop", 0.0)),
        skip_drop=float(params.get("skip_drop", 0.0)),
        sample_type=params.get("sample_type", "uniform"),
        normalize_type=params.get("normalize_type", "tree"),

        early_stopping_rounds=EARLY_STOP,
    )
    model.fit(X_tr_np, y_tr, eval_set=[(X_va_np, y_va)], verbose=False)
    return model

def build_lgb(params: dict):

    model = lgb.LGBMClassifier(
        objective="binary",
        boosting_type=params["boosting_type"],     # "gbdt" | "dart" | "goss"
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,

        n_estimators=int(params["n_estimators"]),
        learning_rate=float(params["learning_rate"]),

        max_depth=int(params["max_depth"]),
        num_leaves=int(params["num_leaves"]),

        subsample=float(params["subsample"]),
        colsample_bytree=float(params["colsample_bytree"]),

        min_child_samples=int(params["min_child_samples"]),
        min_split_gain=float(params["min_split_gain"]),

        reg_lambda=float(params["reg_lambda"]),
        reg_alpha=float(params["reg_alpha"]),

        # imbalance
        scale_pos_weight=float(params["scale_pos_weight"]),
    )
    model.fit(
        X_tr_np, y_tr,
        eval_set=[(X_va_np, y_va)],
        eval_metric=params["eval_metric"],
        callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOP, verbose=False)]
    )
    return model

def build_cat(params: dict):
    cb_params = dict(
        loss_function="Logloss",
        eval_metric=params["eval_metric"],
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,

        iterations=int(params["iterations"]),
        depth=int(params["depth"]),
        learning_rate=float(params["learning_rate"]),
        l2_leaf_reg=float(params["l2_leaf_reg"]),

        boosting_type=params["boosting_type"],
        bootstrap_type=params["bootstrap_type"],

        rsm=float(params["rsm"]),
        random_strength=float(params["random_strength"]),

        auto_class_weights=params["auto_class_weights"],
        class_weights=params["class_weights"],

        od_type="Iter",
        od_wait=EARLY_STOP,
    )

    # Prevent incompatible weight settings
    if cb_params.get("auto_class_weights") is not None:
        cb_params["class_weights"] = None

    # Bayesian bootstrap: allow bagging_temperature, disallow subsample
    if cb_params.get("bootstrap_type") == "Bayesian":
        cb_params["bagging_temperature"] = float(params["bagging_temperature"])
    else:
        cb_params["subsample"] = float(params["subsample"])

    model = CatBoostClassifier(**cb_params)
    model.fit(X_tr_np, y_tr, eval_set=(X_va_np, y_va), use_best_model=True)
    return model

def fit_model(kind: str, params: dict):
    if kind == "xgb":
        return build_xgb(params)
    if kind == "lgb":
        return build_lgb(params)
    if kind == "cat":
        return build_cat(params)
    raise ValueError("Unknown kind")


XGB_SPACE_LOGLOSS = dict(
    booster=["gbtree"],
    eval_metric=["logloss"],
    n_estimators=[3000, 6000],
    max_depth=[3, 4, 5],
    learning_rate=[0.02, 0.03, 0.05],
    subsample=[0.8, 0.9, 1.0],
    colsample_bytree=[0.6, 0.8, 1.0],
    min_child_weight=[1, 2, 5],
    gamma=[0.0, 0.5, 1.0],
    reg_lambda=[1.0, 2.0, 5.0],
    reg_alpha=[0.0, 0.1, 0.5],
    scale_pos_weight=[scale_pos_weight],
    max_delta_step=[0.0, 1.0, 5.0],
)

XGB_SPACE_AUCPR = dict(**XGB_SPACE_LOGLOSS)
XGB_SPACE_AUCPR["eval_metric"] = ["aucpr"]

XGB_SPACE_DART = dict(**XGB_SPACE_AUCPR)
XGB_SPACE_DART["booster"] = ["dart"]
XGB_SPACE_DART.update(dict(
    rate_drop=[0.05, 0.10, 0.20],
    skip_drop=[0.0, 0.25, 0.50],
    sample_type=["uniform"],
    normalize_type=["tree"],
))


LGB_SPACE_GBDT = dict(
    boosting_type=["gbdt"],
    eval_metric=["binary_logloss"],
    n_estimators=[5000, 12000],
    learning_rate=[0.01, 0.02, 0.03],
    max_depth=[-1, 6, 10],
    num_leaves=[31, 63, 127, 255],
    subsample=[0.8, 0.9, 1.0],
    colsample_bytree=[0.7, 0.85, 1.0],
    min_child_samples=[10, 20, 40],
    min_split_gain=[0.0, 0.05, 0.1],
    reg_lambda=[0.0, 1.0, 3.0],
    reg_alpha=[0.0, 0.1, 0.3],
    scale_pos_weight=[scale_pos_weight],
)

LGB_SPACE_AUC = dict(**LGB_SPACE_GBDT)
LGB_SPACE_AUC["eval_metric"] = ["auc"]

LGB_SPACE_DART = dict(**LGB_SPACE_AUC)
LGB_SPACE_DART["boosting_type"] = ["dart"]

LGB_SPACE_GOSS = dict(**LGB_SPACE_AUC)
LGB_SPACE_GOSS["boosting_type"] = ["goss"]
LGB_SPACE_GOSS["subsample"] = [1.0]

CAT_SPACE_BASE = dict(
    eval_metric=["AUC"],
    iterations=[12000, 20000],
    depth=[4, 6, 8, 10],
    learning_rate=[0.01, 0.02, 0.03, 0.05],
    l2_leaf_reg=[1.0, 3.0, 5.0, 10.0],
    boosting_type=["Plain", "Ordered"],
    bootstrap_type=["Bayesian"],
    subsample=[0.8, 0.9, 1.0],
    rsm=[0.7, 0.85, 1.0],
    random_strength=[1.0, 5.0, 10.0],
    bagging_temperature=[0.0, 1.0, 2.0],
)

GRID_XGB_LOGLOSS = sample_param_space(XGB_SPACE_LOGLOSS, n_samples=18, seed=RANDOM_STATE + 10)
GRID_XGB_AUCPR   = sample_param_space(XGB_SPACE_AUCPR,   n_samples=18, seed=RANDOM_STATE + 11)
GRID_XGB_DART    = sample_param_space(XGB_SPACE_DART,    n_samples=18, seed=RANDOM_STATE + 12)

GRID_LGB_GBDT    = sample_param_space(LGB_SPACE_GBDT,    n_samples=22, seed=RANDOM_STATE + 20)
GRID_LGB_AUC     = sample_param_space(LGB_SPACE_AUC,     n_samples=22, seed=RANDOM_STATE + 21)
GRID_LGB_DART    = sample_param_space(LGB_SPACE_DART,    n_samples=18, seed=RANDOM_STATE + 22)
GRID_LGB_GOSS    = sample_param_space(LGB_SPACE_GOSS,    n_samples=18, seed=RANDOM_STATE + 23)

# CatBoost grids with different imbalance handling
GRID_CAT_NONE = sample_param_space({**CAT_SPACE_BASE,
                                    "auto_class_weights":[None],
                                    "class_weights":[None]}, n_samples=24, seed=RANDOM_STATE + 30)

GRID_CAT_BAL  = sample_param_space({**CAT_SPACE_BASE,
                                    "auto_class_weights":["Balanced"],
                                    "class_weights":[None]}, n_samples=24, seed=RANDOM_STATE + 31)

CW1 = float(scale_pos_weight)
GRID_CAT_CW   = sample_param_space({**CAT_SPACE_BASE,
                                    "auto_class_weights":[None],
                                    "class_weights":[[1.0, CW1],
                                                    [1.0, 0.75*CW1],
                                                    [1.0, 1.25*CW1]]}, n_samples=24, seed=RANDOM_STATE + 32)

VARIANTS = [
    {"name": "XGBoost | gbtree | logloss-ES", "kind": "xgb", "grid": GRID_XGB_LOGLOSS},
    {"name": "XGBoost | gbtree | aucpr-ES",   "kind": "xgb", "grid": GRID_XGB_AUCPR},
    {"name": "XGBoost | dart   | aucpr-ES",   "kind": "xgb", "grid": GRID_XGB_DART},

    {"name": "LightGBM | gbdt | logloss-ES",  "kind": "lgb", "grid": GRID_LGB_GBDT},
    {"name": "LightGBM | gbdt | auc-ES",      "kind": "lgb", "grid": GRID_LGB_AUC},
    {"name": "LightGBM | dart | auc-ES",      "kind": "lgb", "grid": GRID_LGB_DART},
    {"name": "LightGBM | goss | auc-ES",      "kind": "lgb", "grid": GRID_LGB_GOSS},

    {"name": "CatBoost | none     | AUC-ES",  "kind": "cat", "grid": GRID_CAT_NONE},
    {"name": "CatBoost | balanced | AUC-ES",  "kind": "cat", "grid": GRID_CAT_BAL},
    {"name": "CatBoost | class_w  | AUC-ES",  "kind": "cat", "grid": GRID_CAT_CW},
]

print("Total variants:", len(VARIANTS))
print("Total fits (Youden + F1):", 2 * sum(len(v["grid"]) for v in VARIANTS))


def get_val_probs(model):
    return model.predict_proba(X_va_np)[:, 1]

def get_test_probs(model):
    return model.predict_proba(X_te_np)[:, 1]

def search_best_on_val(variant, criterion: str, thr_grid_size: int = 500):
    best_score = None
    best = None

    for params in variant["grid"]:
        model = fit_model(variant["kind"], params)
        p_va = get_val_probs(model)

        pick = pick_threshold(y_va, p_va, criterion=criterion, grid_size=thr_grid_size)
        score = pick["score"]

        if (best_score is None) or (score > best_score):
            best_score = score
            best = {"model": model, "params": params, "thr": float(pick["thr"])}

    return best

def eval_threshold_metrics_on_test(model, thr: float):
    p_te = get_test_probs(model)
    eps = 1e-15
    p_te = np.clip(p_te, eps, 1 - eps)
    sens, spec, prec, acc, f1, youden = _metrics_from_probs(y_te, p_te, thr)
    return {
        "Sensitivity": float(sens),
        "Specificity": float(spec),
        "Precision":   float(prec),
        "Accuracy":    float(acc),
    }

def eval_probability_metrics_on_test(model):
    p_te = get_test_probs(model)
    eps = 1e-15
    p_te = np.clip(p_te, eps, 1 - eps)
    return {
        "ROC_AUC": float(roc_auc_score(y_te, p_te)),
        "PR_AUC":  float(average_precision_score(y_te, p_te)),
        "Brier":   float(brier_score_loss(y_te, p_te)),
        "LogLoss": float(log_loss(y_te, p_te)),
    }

def run_and_build_tables():
    rows_youden = []
    rows_f1 = []
    rows_prob = []

    for v in VARIANTS:
        print(f"\n[VAL search] {v['name']}  (grid={len(v['grid'])})")

        # Youden selection
        sel_y = search_best_on_val(v, criterion="youden", thr_grid_size=600)
        met_y = eval_threshold_metrics_on_test(sel_y["model"], sel_y["thr"])
        rows_youden.append({
            "Variant": v["name"],
            "Threshold": sel_y["thr"],
            **met_y,
            "_params": sel_y["params"],
        })

        # F1 selection
        sel_f = search_best_on_val(v, criterion="f1", thr_grid_size=600)
        met_f = eval_threshold_metrics_on_test(sel_f["model"], sel_f["thr"])
        rows_f1.append({
            "Variant": v["name"],
            "Threshold": sel_f["thr"],
            **met_f,
            "_params": sel_f["params"],
        })

        # Probability metrics: use Youden-selected best model per variant
        prob = eval_probability_metrics_on_test(sel_y["model"])
        rows_prob.append({
            "Variant": v["name"],
            **prob,
            "_params": sel_y["params"],
        })

    df_y = pd.DataFrame(rows_youden).drop(columns=["_params"])
    df_f = pd.DataFrame(rows_f1).drop(columns=["_params"])
    df_p = pd.DataFrame(rows_prob).drop(columns=["_params"])
    return df_y, df_f, df_p

df_test_youden, df_test_f1, df_prob = run_and_build_tables()

print("\nTEST — evaluated using VAL-picked (hyperparams, threshold) from Youden selection:")
display(style_table_viridis(df_test_youden))

print("\nTEST — evaluated using VAL-picked (hyperparams, threshold) from F1 selection:")
display(style_table_viridis(df_test_f1))

print("\nTEST — probability metrics (threshold-free), using Youden-picked best model per VARIANT:")
display(style_prob_table_viridis(df_prob))


Train size: 1689 | pos: 335 | neg: 1354 | scale_pos_weight: 4.042
Total variants: 10
Total fits (Youden + F1): 412

[VAL search] XGBoost | gbtree | logloss-ES  (grid=18)


/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [17:07:01] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "normalize_type", "rate_drop", "sample_type", "skip_drop" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [17:07:02] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "normalize_type", "rate_drop", "sample_type", "skip_drop" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [17:07:03] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "normalize_type", "rate_drop", "sample_type", "skip_drop" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarni


[VAL search] XGBoost | gbtree | aucpr-ES  (grid=18)


/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [17:07:39] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "normalize_type", "rate_drop", "sample_type", "skip_drop" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [17:07:39] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "normalize_type", "rate_drop", "sample_type", "skip_drop" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [17:07:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "normalize_type", "rate_drop", "sample_type", "skip_drop" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarni


[VAL search] XGBoost | dart   | aucpr-ES  (grid=18)

[VAL search] LightGBM | gbdt | logloss-ES  (grid=22)


...

Variant,Threshold,Sensitivity,Specificity,Precision,Accuracy
XGBoost | gbtree | logloss-ES,0.256,0.857,0.958,0.835,0.938
XGBoost | gbtree | aucpr-ES,0.364,0.848,0.922,0.731,0.908
XGBoost | dart | aucpr-ES,0.328,0.857,0.933,0.762,0.918
LightGBM | gbdt | logloss-ES,0.222,0.866,0.960,0.843,0.941
LightGBM | gbdt | auc-ES,0.250,0.866,0.967,0.866,0.947
LightGBM | dart | auc-ES,0.299,0.839,0.969,0.870,0.943
LightGBM | goss | auc-ES,0.278,0.866,0.916,0.719,0.906
CatBoost | none | AUC-ES,0.132,0.875,0.942,0.790,0.929
CatBoost | balanced | AUC-ES,0.396,0.848,0.953,0.819,0.933
CatBoost | class_w | AUC-ES,0.328,0.875,0.951,0.817,0.936



TEST — evaluated using VAL-picked (hyperparams, threshold) from F1 selection:


Variant,Threshold,Sensitivity,Specificity,Precision,Accuracy
XGBoost | gbtree | logloss-ES,0.475,0.821,0.978,0.902,0.947
XGBoost | gbtree | aucpr-ES,0.551,0.812,0.976,0.892,0.943
XGBoost | dart | aucpr-ES,0.633,0.777,0.978,0.897,0.938
LightGBM | gbdt | logloss-ES,0.346,0.812,0.976,0.892,0.943
LightGBM | gbdt | auc-ES,0.702,0.768,0.980,0.905,0.938
LightGBM | dart | auc-ES,0.556,0.812,0.984,0.929,0.950
LightGBM | goss | auc-ES,0.385,0.804,0.971,0.874,0.938
CatBoost | none | AUC-ES,0.164,0.821,0.978,0.902,0.947
CatBoost | balanced | AUC-ES,0.377,0.804,0.980,0.909,0.945
CatBoost | class_w | AUC-ES,0.518,0.777,0.980,0.906,0.940



TEST — probability metrics (threshold-free), using Youden-picked best model per VARIANT:


Variant,ROC_AUC,PR_AUC,Brier,LogLoss
XGBoost | gbtree | logloss-ES,0.9474,0.8923,0.0464,0.1838
XGBoost | gbtree | aucpr-ES,0.9380,0.8723,0.0648,0.2535
XGBoost | dart | aucpr-ES,0.9354,0.8755,0.0563,0.2199
LightGBM | gbdt | logloss-ES,0.9439,0.8904,0.0469,0.1847
LightGBM | gbdt | auc-ES,0.9505,0.8951,0.0460,0.1778
LightGBM | dart | auc-ES,0.9512,0.9032,0.0430,0.1939
LightGBM | goss | auc-ES,0.9476,0.8860,0.0519,0.2013
CatBoost | none | AUC-ES,0.9443,0.8915,0.0497,0.1900
CatBoost | balanced | AUC-ES,0.9498,0.9055,0.0513,0.2083
CatBoost | class_w | AUC-ES,0.9541,0.9101,0.0479,0.1939
